# 03-5. 함수와 스코프 실습

이 노트북은 03-4의 반복·집계 코드를 검증 가능한 함수로 나눕니다. 각 셀은 **결과 예측 → 실행 → 설명 → 입력 변경** 순서로 학습하세요. 파일·모듈·예외 처리는 이후 절에서 다룹니다.

## 0. `print()`와 `return` 예측

`shown`과 `returned`에 저장될 값을 먼저 적어 보세요.

In [ ]:
def show_action():
    print("DENY")

def get_action():
    return "DENY"

shown = show_action()
returned = get_action()

print("shown:", shown)
print("returned:", returned)
assert shown is None
assert returned == "DENY"

## 1. 입력·처리·출력 계약

함수를 읽을 때 매개변수, 처리 규칙, 반환값, 외부 상태 변경 여부를 찾습니다.

In [ ]:
def classify_count(count, threshold):
    if count >= threshold:
        return "WARNING"
    return "NORMAL"

print(classify_count(5, 3))
print(classify_count(2, 3))
assert classify_count(3, 3) == "WARNING"

## 2. 조기 반환과 반환 형태

잘못된 입력과 허용되지 않은 값을 먼저 반환하면 정상 흐름의 들여쓰기를 줄일 수 있습니다. 모든 경로가 `str` 또는 `None`이라는 일관된 의미를 갖는지 확인하세요.

In [ ]:
def normalize_action(action):
    if not isinstance(action, str):
        return None

    normalized = action.strip().upper()
    if normalized not in {"ALLOW", "DENY"}:
        return None

    return normalized

assert normalize_action(" deny ") == "DENY"
assert normalize_action("BLOCK") is None
assert normalize_action(443) is None

## 3. 여러 값 반환

쉼표로 반환한 여러 값은 튜플이며 호출부에서 언패킹할 수 있습니다.

In [ ]:
def count_actions(actions):
    allow_count = actions.count("ALLOW")
    deny_count = actions.count("DENY")
    return allow_count, deny_count

counts = count_actions(["ALLOW", "DENY", "DENY"])
allow_count, deny_count = counts

print(counts, type(counts))
assert counts == (1, 2)
assert allow_count == 1 and deny_count == 2

## 4. 위치·키워드·기본값·키워드 전용 인자

`*` 뒤의 매개변수는 키워드로만 전달합니다. 숫자 인자의 의미와 순서를 호출부에 드러냅니다.

In [ ]:
def classify(count, *, warning=3, critical=5):
    if count >= critical:
        return "CRITICAL"
    if count >= warning:
        return "WARNING"
    return "NORMAL"

assert classify(2) == "NORMAL"
assert classify(4, warning=3, critical=5) == "WARNING"
assert classify(count=7, warning=4, critical=6) == "CRITICAL"

## 5. `*args`, `**kwargs`, 호출 시 펼치기

정의의 `*args`·`**kwargs`는 값을 모으고, 호출의 `*`·`**`는 컬렉션을 인자로 펼칩니다.

In [ ]:
def total_attempts(*counts):
    return sum(counts)

def build_record(action, **fields):
    record = {"action": action}
    record.update(fields)
    return record

attempt_values = [1, 2, 3]
event_fields = {"ip": "198.51.100.9", "port": 443}

attempt_total = total_attempts(*attempt_values)
event = build_record("DENY", **event_fields)

print(attempt_total)
print(event)
assert attempt_total == 6
assert event == {"action": "DENY", "ip": "198.51.100.9", "port": 443}

## 6. 변경 가능한 기본값

먼저 잘못된 함수에서 같은 기본 리스트가 공유되는지 관찰하고, `None` 표식으로 호출마다 새 리스트를 만듭니다.

In [ ]:
def add_tag_bad(tag, tags=[]):
    tags.append(tag)
    return tags

bad_first = add_tag_bad("web")
bad_second = add_tag_bad("critical")

print(bad_first)
print(bad_second)
assert bad_first is bad_second
assert bad_second == ["web", "critical"]

In [ ]:
def add_tag(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

first_tags = add_tag("web")
second_tags = add_tag("critical")

assert first_tags == ["web"]
assert second_tags == ["critical"]
assert first_tags is not second_tags

## 7. 재할당과 내부 변경

매개변수를 새 리스트에 재할당하면 원본 이름은 바뀌지 않습니다. 전달받은 리스트에 `append()`하면 원본 객체가 변경됩니다. 새 결과를 반환하는 방식과 비교하세요.

In [ ]:
def replace_events(events):
    events = [{"action": "REPLACED"}]
    return events

def append_event(events, event):
    events.append(event)

def with_event(events, event):
    new_events = events.copy()
    new_events.append(event)
    return new_events

original = [{"action": "ALLOW"}]
replacement = replace_events(original)
assert original == [{"action": "ALLOW"}]
assert replacement == [{"action": "REPLACED"}]

mutated = []
append_event(mutated, {"action": "DENY"})
assert mutated == [{"action": "DENY"}]

source = []
copied_result = with_event(source, {"action": "DENY"})
assert source == []
assert copied_result == [{"action": "DENY"}]

## 8. 스코프와 상태

지역 이름은 함수 밖에서 보이지 않습니다. `nonlocal`은 중첩 함수가 바깥 함수의 상태를 갱신할 때 사용하지만, 일반 함수는 입력과 반환값을 우선합니다.

In [ ]:
status = "GLOBAL"

def get_local_status():
    status = "LOCAL"
    return status

assert get_local_status() == "LOCAL"
assert status == "GLOBAL"

def make_counter():
    count = 0

    def increment():
        nonlocal count
        count += 1
        return count

    return increment

counter = make_counter()
assert counter() == 1
assert counter() == 2

## 9. 함수 객체와 콜백

판단 함수를 다른 함수에 전달하면 선택 규칙을 재사용할 수 있습니다. 함수 자체 `is_denied`와 호출 결과 `is_denied(event)`를 구분하세요.

In [ ]:
def is_denied(event):
    return event["action"] == "DENY"

def select_events(events, rule):
    selected = []
    for event in events:
        if rule(event):
            selected.append(event)
    return selected

callback_events = [
    {"action": "ALLOW", "port": 443},
    {"action": "DENY", "port": 22},
    {"action": "DENY", "port": 443},
]

denied_events = select_events(callback_events, is_denied)
https_events = select_events(callback_events, lambda event: event["port"] == 443)

assert len(denied_events) == 2
assert len(https_events) == 2

## 10. 재귀의 기저 조건

자기 호출 전에 종료 조건이 있는지, 인자가 종료 조건에 가까워지는지 확인합니다.

In [ ]:
def countdown(number):
    if number <= 0:
        return ["done"]
    return [number] + countdown(number - 1)

assert countdown(3) == [3, 2, 1, "done"]
assert countdown(0) == ["done"]

## 11. 경계값으로 계약 검증

`bool`이 `int`의 하위 자료형이라는 점까지 포함해 포트 검증 함수를 시험합니다.

In [ ]:
def is_valid_port(port):
    return type(port) is int and 1 <= port <= 65535

assert is_valid_port(0) is False
assert is_valid_port(1) is True
assert is_valid_port(65535) is True
assert is_valid_port(65536) is False
assert is_valid_port("443") is False
assert is_valid_port(True) is False
print("포트 경계값 검증 통과")

## 12. 미니 실습: 이벤트 분석 함수 분리

검증, 분류, 집계, 보고 문자열 생성을 서로 다른 함수로 나눕니다. 집계 함수는 입력 컬렉션을 바꾸지 않고 새 결과를 반환합니다.

In [ ]:
def is_valid_event(event: dict) -> bool:
    """필수 필드와 action·ip·port 값이 유효한지 반환한다."""
    action = event.get("action")
    ip = event.get("ip")
    port = event.get("port")

    has_valid_action = action in {"ALLOW", "DENY"}
    has_valid_ip = isinstance(ip, str) and bool(ip.strip())
    has_valid_port = type(port) is int and 1 <= port <= 65535
    return has_valid_action and has_valid_ip and has_valid_port

def classify_event(event: dict, *, sensitive_ports=(22, 3389)) -> str:
    """유효한 이벤트를 ALLOW, DENY, CRITICAL 중 하나로 분류한다."""
    if event["action"] == "DENY" and event["port"] in sensitive_ports:
        return "CRITICAL"
    if event["action"] == "DENY":
        return "DENY"
    return "ALLOW"

def summarize_events(events: list[dict]) -> dict:
    """이벤트를 검증·분류하고 건수와 IP별 DENY 횟수를 반환한다."""
    counts = {
        "valid": 0,
        "invalid": 0,
        "allow": 0,
        "deny": 0,
        "critical": 0,
    }
    deny_count_by_ip = {}

    for event in events:
        if not is_valid_event(event):
            counts["invalid"] += 1
            continue

        counts["valid"] += 1
        level = classify_event(event)

        if level == "ALLOW":
            counts["allow"] += 1
        else:
            counts["deny"] += 1
            ip = event["ip"]
            deny_count_by_ip[ip] = deny_count_by_ip.get(ip, 0) + 1

            if level == "CRITICAL":
                counts["critical"] += 1

    return {
        "counts": counts,
        "deny_count_by_ip": deny_count_by_ip,
    }

def format_summary(summary: dict) -> str:
    """집계 딕셔너리를 한 줄 보고 문자열로 변환한다."""
    counts = summary["counts"]
    return (
        f"valid={counts['valid']} "
        f"invalid={counts['invalid']} "
        f"deny={counts['deny']} "
        f"critical={counts['critical']}"
    )

In [ ]:
events = [
    {"action": "ALLOW", "ip": "10.0.0.5", "port": 443},
    {"action": "DENY", "ip": "198.51.100.9", "port": 22},
    {"action": "DENY", "ip": "198.51.100.9", "port": 3389},
    {"action": "BLOCK", "ip": "203.0.113.10", "port": 70000},
    {"action": "DENY", "ip": "203.0.113.10", "port": 443},
]
events_before = [event.copy() for event in events]

summary = summarize_events(events)
report = format_summary(summary)

print(summary)
print(report)
assert events == events_before

In [ ]:
expected_summary = {
    "counts": {
        "valid": 4,
        "invalid": 1,
        "allow": 1,
        "deny": 3,
        "critical": 2,
    },
    "deny_count_by_ip": {
        "198.51.100.9": 2,
        "203.0.113.10": 1,
    },
}

assert summary == expected_summary
assert report == "valid=4 invalid=1 deny=3 critical=2"
assert summarize_events([])["counts"] == {
    "valid": 0,
    "invalid": 0,
    "allow": 0,
    "deny": 0,
    "critical": 0,
}
print("이벤트 분석 함수 검증 통과")

## 13. 확장 실습: 포트별 집계

유효한 이벤트만 포트별로 집계하는 단일 책임 함수를 작성하고 빈 입력·잘못된 입력·정상 입력을 검증합니다.

In [ ]:
def count_events_by_port(events):
    """유효한 이벤트의 포트별 횟수를 새 딕셔너리로 반환한다."""
    counts = {}
    for event in events:
        if is_valid_event(event):
            port = event["port"]
            counts[port] = counts.get(port, 0) + 1
    return counts

assert count_events_by_port([]) == {}
assert count_events_by_port([
    {"action": "BLOCK", "ip": "", "port": 70000},
]) == {}
assert count_events_by_port(events) == {443: 2, 22: 1, 3389: 1}
print("확장 실습 검증 통과")

## 14. 자기 점검

1. `return`을 `print()`로 바꾸면 미니 실습의 `assert`가 왜 불가능해지나요?
2. `sensitive_ports=[]`를 기본값으로 사용하지 않은 이유는 무엇인가요?
3. `summarize_events()`가 입력 목록을 변경하지 않았음을 어떻게 확인했나요?
4. `classify_event(event, (22, 3389))` 호출이 허용되지 않는 이유는 무엇인가요?
5. 타입 힌트가 있어도 `is_valid_event()` 안의 조건 검사가 필요한 이유는 무엇인가요?
6. `select_events(callback_events, is_denied)`에서 괄호를 붙이지 않은 이유를 설명하세요.